# Quantum Teleportation

## Introduction

The Quantum Teleportation algorithm is an algorithm to teleport quantum information ($\psi$) from a sender (Alice) at one location to a receiver (Bob) some distance away.

In this exercise, we will use the Quantum Teleportation algorithm to:
- Implement an hybrid circuit with measure and classical control
- Evaluate the circuit into an HPS with automatic and interactive rewriting
- Verify that the state of $bob$ at the end of the circuit is the input state $\psi$
- Do all the above again with quantum registers of several qubits

## Circuit

![Teleportation circuit](../img/teleportation.png)

**Notes:**
- The | at the begining of a register means that it is initialized to 0
- The arrow is a measure

## Single qubit registers

### Circuit implementation

Implement the circuit.  
We can see the measure operator **-@**.

In [ ]:
#require "hqbricks"
open Hqbricks
open Gate_set_impl.Clifford_k

let tp_circ =
  let open Prog in
  let psi = qreg "psi" ~$1 in
  let alice = qreg "alice" ~$1 in
  let bob = qreg "bob" ~$1 in
  let m_psi = creg "m_psi" ~$1 in
  let m_alice = creg "m_alice" ~$1 in
  init_qreg alice
  (* TODO: missing instruction *)
  -- (alice |> h)
  -- (qbit_val alice ~$0 => (bob |> x))
  (* TODO: missing instruction *)
  -- (psi -@ m_psi)
  (* TODO: missing instruction *)
  -- (cbit_val m_psi ~$0 => (bob |> z))

let () =
  print_endline "\nCirc:";
  Prog.print tp_circ

### HPS evaluation and rewrite

First, let's look at the evaluated HPS without any rewrite:

In [ ]:
let input_hps = Hps.add_qmem_vec_x ("psi", 0) 1 0 Hps.one
let () =
  print_endline "\nInput HPS:";
  print_endline (Hps.to_string input_hps);
  print_endline ""

let hps = Evaluator.(evaluate_prog tp_circ input_hps ~print:false)
let () =
  print_endline "\nHPS:";
  print_endline (Hps.to_string hps)

Here, we would like to apply the **Discard rule** to discard all registers except $bob$ and then check if the resulting HPS only contains $x_0$ in $bob_0$.  
The issue is that we cannot discard $m\_alice$ as is because the discarded HPS cannot contain input variables and $m\_alice$ contains $x_0$.  
In order to remove $x_0$ from $m\_alice$, we can use the **Change-Var** rule to replace $y_0$ by $y_0 \oplus x_0$ in $m\_alice$, we will then have $y_0 \oplus x_0 \oplus x_0 = y_0$ and we will be able to discard.  

**Exercise**:  
We will try to evaluate the teleportation circuit with interactive rewrite and verify the resulting HPS.
Open a terminal by clicking File -> New -> Terminal then run `cd examples/teleportation && dune build && cd _build/default/bin` and `./main.exe`. You can then see the evaluation step by step (press enter to go to next step) while being able to run rewrite commands at each step.  
The goal of this exercise is to validate the verification by using **change_var** and **discard** commands at the last step of the evaluation.

It is also possible to apply **automatic rewrite** (Change-Var and HH) when calling `evaluate_prog` by setting `~rewrite_settings:all_auto`:

In [ ]:
let hps = Evaluator.(evaluate_prog tp_circ input_hps ~rewrite_settings:all_auto)
let () =
  print_endline "\nHPS:";
  print_endline (Hps.to_string hps)

### Verification

We will now verify that this HPS satisfies the spec HPS which is the input HPS but with $bob$ instead of $\psi$ using the **Assertion.hps_satifies** function. This function will try to discard every registers that is not in the spec, then check the equivalence between the spec HPS and the HPS.

In [ ]:
let spec_hps = Hps.add_qmem_vec_x ("bob", 0) 1 0 Hps.one in
assert (Assertion.hps_satisfies spec_hps hps)

## Multi-qubit registers

### Circuit implementation

Implement the Teleportation circuit with registers $\psi$, $alice$ and $bob$ of 100 qubits.  
The controls must be replaced by a loop of controls, for example: for i from 0 to 99 (included) do if $alice_i$ then $bob_i$ |> $X$ else $Skip$ done, this can be done using the **>>** operator.

In [ ]:
let tp_circ_multi =
  let open Prog in
  let reg_len = (* TODO: fill len *) in
  let psi = qreg "psi" reg_len in
  let alice = qreg "alice" reg_len in
  let bob = qreg "bob" reg_len in
  let m_psi = creg "m_psi" reg_len in
  let m_alice = creg "m_alice" reg_len in
  init_qreg alice
  (* TODO: missing insctruction *)
  -- (alice |> h)
  -- ((alice, "i") >> (q_idxv bob "i" |> x))
  (* TODO: missing insctruction *)
  -- (psi -@ m_psi)
  (* TODO: missing insctruction *)
  -- C.((m_alice, "i") >> (q_idxv bob "i" |> x))
  (* TODO: missing insctruction *)

let () =
  print_endline "\nCirc:";
  Prog.print tp_circ_multi

### HPS evaluation and rewrite

Evaluate the 100-qubits Teleportation circuit with **~rewrite_settings:all_auto** and **~print:false** (this one is important as there would be a LOT of print otherwise).

In [ ]:
let input_hps_multi = Hps.add_qmem_vec_x ("psi", 0) 100 0 Hps.one
let () =
  print_endline "\nInput HPS:";
  print_endline (Hps.to_string input_hps_multi);
  print_endline ""

let hps_multi = Evaluator.(evaluate_prog (* TODO: parameters *) ~print:false)
let () =
  print_endline "\nHPS:";
  print_endline (Hps.to_string hps_multi)

### Verification

Verify the 100-qubit Teleportation circuit.

In [ ]:
let spec_hps = (* TODO: spec HPS *) in
assert (Assertion.hps_satisfies spec_hps hps_multi)